# Razvilka: evaluate a model

[Razvilka](https://huggingface.co/datasets/artemsnegirev/razvilka) is a Russian benchmark of
**structured decisions over text**: 735 items, 15 tasks, four question types (`choice`, `noul`
yes/no, `score`, `ranking`). Each item is a text plus one question with a closed set of options;
the metric is accuracy.

This notebook loads the set, shows a few items of each type, runs
[FRIDA-Decisions](https://huggingface.co/ai-forever/FRIDA-Decisions), scores it next to chance and a
lexical baseline, and ends with a template for evaluating your own model.

On Colab, a GPU runtime (Runtime → Change runtime type → T4 GPU) makes the model cells fast; everything
else runs on CPU.

In [ ]:
!pip install -q datasets pandas "frida-decisions @ git+https://github.com/ai-forever/frida-decisions"

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import urllib.request

REPO = "artemsnegirev/razvilka"
# The scorer ships with the dataset: a single file, standard library only.
EVAL_URL = "https://raw.githubusercontent.com/ai-forever/frida-decisions/main/benchmarks/razvilka/razvilka_eval.py"
urllib.request.urlretrieve(EVAL_URL, "razvilka_eval.py")
sys.path.insert(0, ".")
import razvilka_eval as rz

items = rz.load()          # from the Hub; rz.load("data/test.jsonl") reads a local copy
print(len(items), "items")
pd.Series([it["type"] for it in items]).value_counts()

## What an item looks like

`request` is the full request: the text (`state`) and one question with its `type`, `instructions`
and `criteria` (the options, described in words). `gold` is an option key for `choice` and
`ranking` (the best candidate), `true`/`false` for `noul`, and a level index for `score`.

In [ ]:
def show(item, width=300):
    qid, q = rz.question(item)
    state = item["request"]["state"]
    print(f"[{item['type']}] {item['task']}  id={item['id']}")
    print("text:    ", state[:width] + ("…" if len(state) > width else ""))
    print("question:", q["instructions"])
    criteria = q["criteria"]
    pairs = enumerate(criteria) if isinstance(criteria, list) else criteria.items()
    for key, text in pairs:
        text = str(text)
        print(f"   {key}: {text[:120]}{'…' if len(text) > 120 else ''}")
    print("gold:    ", rz.gold_key(item))
    print()

for kind in rz.TYPES:
    for item in [it for it in items if it["type"] == kind][:2]:
        show(item)

## Run FRIDA-Decisions

The requests are passed to the model unchanged. The answer to each question is an object with
`probabilities` (or `noul`, the probability of yes), and the scorer reads it directly.

In [ ]:
import torch
from frida_decisions import Judge

device = "cuda" if torch.cuda.is_available() else "cpu"
# The model reads up to 512 tokens of the text; the default is 384.
judge = Judge.from_pretrained("ai-forever/FRIDA-Decisions", device=device, state_max=512)

def run_judge(items, batch=16):
    predictions = {}
    for start in range(0, len(items), batch):
        chunk = items[start:start + batch]
        responses = judge.judge_batch([it["request"] for it in chunk])
        for item, response in zip(chunk, responses):
            qid, _ = rz.question(item)
            predictions[item["id"]] = response["answers"][qid]
    return predictions

frida_predictions = run_judge(items)
frida = rz.score(frida_predictions, items)
print(rz.format_report(frida))

## Baselines

* **chance**: uniform over each item's options (the `chance` column of every result).
* **lexical**: no model; each option is scored by IDF-weighted word overlap with the text, and the
  best-scoring option wins. A model that does not clear it is matching words, not deciding.

Argmax ties are broken lexicographically by option key, for every system.

In [ ]:
lexical = rz.score(rz.lexical_baseline(items), items)
results = {"FRIDA-Decisions": frida, "lexical": lexical}

def table(results, level=None):
    """Accuracy of every system side by side: overall, per_type or per_task."""
    blocks = {name: (r[level] if level else {"all": r}) for name, r in results.items()}
    first = next(iter(blocks.values()))
    rows = []
    for key, block in first.items():
        row = {"": key, "n": block["n"]}
        if level == "per_task":
            row["type"] = block["type"]
        row.update({name: round(b[key]["accuracy"], 3) for name, b in blocks.items()})
        row["chance"] = round(block["chance"], 3)
        rows.append(row)
    return pd.DataFrame(rows).set_index("").rename_axis(None)

display(table(results))
display(table(results, "per_type"))
display(table(results, "per_task"))

## Evaluate your own model

Write an adapter that takes one request and returns, per question id, either a decided answer or a
distribution:

| type | decided answer | distribution |
|---|---|---|
| `choice`, `ranking` | an option key | `{key: probability}`, or a list of keys best first |
| `noul` | `True` / `False` | `p(true)` as a float |
| `score` | a level index (int) | a list of per-level probabilities |

Items with no answer, or with an answer that is not one of the options, count as wrong.

In [ ]:
def predict(request: dict) -> dict:
    """One request in, {question_id: answer} out."""
    answers = {}
    for qid, question in request["questions"].items():
        ...  # call your model here
        answers[qid] = None
    return answers


def evaluate(predict, items):
    predictions = {}
    for item in items:
        qid, _ = rz.question(item)
        predictions[item["id"]] = predict(item["request"]).get(qid)
    return predictions

# FRIDA-Decisions already has this shape:  predict = lambda request: judge(request)["answers"]

### Example: any OpenAI-compatible chat API

The adapter lists the options as `key: description`, asks for one key, and parses it. Set
`OPENAI_API_KEY` (and `OPENAI_BASE_URL` for a non-OpenAI endpoint) in the environment; no key goes
into the notebook.

In [ ]:
import os, re
from openai import OpenAI          # pip install openai

client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL"), api_key=os.environ["OPENAI_API_KEY"])
MODEL = "your-model-name"

PROMPT = """Text:
{state}

Task: {instructions}

Options:
{options}

Reply with the key of the single best option and nothing else."""

def option_lines(question):
    if question["type"] == "noul":
        criteria = question.get("criteria") or {}
        return {"true": criteria.get("true", "yes"), "false": criteria.get("false", "no")}
    criteria = question["criteria"]
    if isinstance(criteria, list):
        return {str(i): text for i, text in enumerate(criteria)}
    return dict(criteria)

def parse_key(reply, keys):
    reply = reply.strip().strip("\"'`.*").strip()
    if reply in keys:
        return reply
    for token in re.findall(r"[\w-]+", reply):
        if token in keys:
            return token
    return None                    # counted as wrong

def llm_predict(request: dict) -> dict:
    answers = {}
    for qid, question in request["questions"].items():
        options = option_lines(question)
        prompt = PROMPT.format(state=request["state"], instructions=question["instructions"],
                               options="\n".join(f"{k}: {v}" for k, v in options.items()))
        reply = client.chat.completions.create(
            model=MODEL, temperature=0, max_tokens=16,
            messages=[{"role": "user", "content": prompt}]).choices[0].message.content or ""
        answers[qid] = parse_key(reply, list(options))
    return answers

In [ ]:
results["my model"] = rz.score(evaluate(llm_predict, items), items)
display(table(results, "per_type"))